In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [6]:
model_name = "google/gemma-3-1b-pt"
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [11]:
seq = "Question: Where capital of France? Answer:"
answer = "Paris"

In [12]:
seq_tokens = tokenizer.encode(f"{tokenizer.bos_token}{seq}", add_special_tokens=False)
answer_tokens = tokenizer.encode(f"{answer}{tokenizer.eos_token}")

In [13]:
prompt_tokens = seq_tokens + answer_tokens

In [25]:
labels_tokens =(len(seq_tokens)*[-100,])+answer_tokens 

In [26]:
labels_tokens

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 2, 50429, 1]

In [16]:
prompt_tokens

[2, 14977, 236787, 10603, 5279, 529, 7001, 236881, 25685, 236787, 2, 50429, 1]

In [17]:
seq_tokens

[2, 14977, 236787, 10603, 5279, 529, 7001, 236881, 25685, 236787]

In [18]:
answer_tokens

[2, 50429, 1]

In [56]:
input_data = [
    {
        "input_ids": torch.tensor(prompt_tokens).to(device),
        "labels": torch.tensor(labels_tokens).to(device),
        "attention_mask": torch.tensor([1]*len(prompt_tokens)).to(device)
    }
]

In [57]:
from torch.utils.data import DataLoader

In [58]:
data_loader = DataLoader(input_data)

In [59]:
len(data_loader)

1

In [60]:
item = next(iter(data_loader))

In [65]:
item

{'input_ids': tensor([[     2,  14977, 236787,  10603,   5279,    529,   7001, 236881,  25685,
          236787,      2,  50429,      1]], device='cuda:0'),
 'labels': tensor([[ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
              2, 50429,     1]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [66]:
output = model(input_ids=item["input_ids"],labels=item["labels"],attention_mask=item["attention_mask"])

In [64]:
output.loss

tensor(14.7744, device='cuda:0', grad_fn=<NllLossBackward0>)

In [99]:
prompt2 = "Paris is capital of"
answer2 = " France"

prompt2_tokens = tokenizer.encode(f"{prompt2}", add_special_tokens=False)
answer2_tokens = tokenizer.encode(f"{answer2}")

In [115]:
answer2_tokens

[2, 7001]

In [100]:
x = prompt2_tokens + answer2_tokens
y =(len(prompt2_tokens)*[-100,])+answer2_tokens 

In [101]:
input_data2 = [
    {
        "input_ids": torch.tensor(x).to(device),
        "labels": torch.tensor(y).to(device),
        "attention_mask": torch.tensor([1]*len(x)).to(device)
    }
]

In [102]:
data_loader_2 = DataLoader(input_data2)

In [103]:
item = next(iter(data_loader_2))

In [104]:
output = model(input_ids=item["input_ids"],labels=item["labels"],attention_mask=item["attention_mask"])

In [113]:
output.logits.shape

torch.Size([1, 6, 262144])

In [116]:
output.logits[0][5][7001]

tensor(12.4075, device='cuda:0', grad_fn=<SelectBackward0>)

In [118]:
import torch.nn.functional as F
a = F.softmax(output.logits, dim=-1)

In [119]:
a[0][5][7001]

tensor(0.0031, device='cuda:0', grad_fn=<SelectBackward0>)

In [122]:
torch.max(a, dim=-1)

torch.return_types.max(
values=tensor([[0.9903, 0.5128, 0.3930, 0.5809, 0.1019, 0.2837]], device='cuda:0',
       grad_fn=<MaxBackward0>),
indices=tensor([[ 50429,   9079,    563,    529,    184, 255999]], device='cuda:0'))

In [129]:
tokenizer.decode([255999])

'<start_of_image>'

In [ ]:
len(output.logits[0][5][])

262144